In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
client = Client()
client

2025-04-15 09:04:00,566 - distributed.preloading - INFO - Creating preload: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py
2025-04-15 09:04:00,567 - distributed.utils - INFO - Reload module schedplugin from .py file
2025-04-15 09:04:00,571 - distributed.preloading - INFO - Import preload module: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py
/home/129/ms5578/miniconda3/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35343 instead
  warnings.warn(


Modifying workers


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/gadi-cpu-bdw-0119.gadi.nci.org.au/23404/proxy/35343/status,
Dashboard: /node/gadi-cpu-bdw-0119.gadi.nci.org.au/23404/proxy/35343/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42975,Workers: 28
Dashboard: /node/gadi-cpu-bdw-0119.gadi.nci.org.au/23404/proxy/35343/status,Total threads: 28
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45667,Total threads: 1
Dashboard: /node/gadi-cpu-bdw-0119.gadi.nci.org.au/23404/proxy/32969/status,Memory: 0 B
Nanny: tcp://127.0.0.1:46865,


In [4]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [8]:
# Selecting Jul 2009 - Jun 2024
fdates = [m.strftime('%Y%m') for m in pd.date_range("20090701", "20240630", freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

In [9]:
tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time', combine='nested', 
                           parallel=True, data_vars='minimal',coords='minimal', 
                           drop_variables = "time_bnds",chunks="auto")

t95_baseline = xr.open_dataarray(f"{workingDir}/data/preprocess/t95_baseline.nc").values

In [11]:
encoding = {"tas":{"zlib": True, "complevel": 4, "shuffle": True}}

tas_ds.to_netcdf(f'{write_path}tas_hwperiod.nc',
                                        encoding=encoding,
                                        engine="netcdf4")